# Relational Databases & Normalization 

This notebook is the SQL-adjacent side of relational database theory: before writing real SQL queries, you need to understand *why* tables are structured the way they are. That "why" is **normalization** — the process of removing data redundancy.

We'll use real SQL (via Python's built-in `sqlite3`) to see a **bad, unnormalized table**, understand exactly what rules it breaks, and then fix it step by step into 1NF and 2NF and 3NF.

## Background: Flat Files vs. Relational Databases

A **Flat File** stores data in one simple structure with no relationships between separate tables (Session 1). This is fine for small, simple data — but it struggles with two things:

- **Integrity** (یکپارچگی) — keeping facts accurate as data changes
- **Consistency** (استحکام) — making sure the same fact isn't stored in two places and allowed to disagree with itself

In **June 1970**, Edgar F. Codd (IBM) published *"A Relational Model of Data for Large Shared Data Banks,"* introducing the **Relational Database** model — data organized into linked tables — specifically to solve these flat-file problems. Codd's later papers introduced **Normalization** and the first three **Normal Forms**.

A **Database** (پایگاه داده) is simply defined as: a collection of organized data.

## Step 1: A Bad, Unnormalized Table

Let's set up a single flat table for a course enrollment system — the kind of table you'd get if you just dumped everything into one spreadsheet without thinking about structure.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# An UNNORMALIZED table: student + course + instructor info all mixed together
cur.execute('''
CREATE TABLE Enrollments_Bad (
    StudentID INTEGER,
    StudentName TEXT,
    CourseID TEXT,
    CourseName TEXT,
    Instructor TEXT,
    Grade TEXT
)
''')

rows = [
    (1, "Ali",    "CS101", "Intro to Programming", "Dr. Smith", "A"),
    (1, "Ali",    "MA201", "Linear Algebra",        "Dr. Jones", "B"),
    (2, "Sara",   "CS101", "Intro to Programming", "Dr. Smith", "A"),
    (3, "Behnam", "CS101", "Intro to Programming", "Dr. Smith", "B"),
]
cur.executemany("INSERT INTO Enrollments_Bad VALUES (?, ?, ?, ?, ?, ?)", rows)
conn.commit()

pd.read_sql("SELECT * FROM Enrollments_Bad", conn)

,StudentID,StudentName,CourseID,CourseName,Instructor,Grade
0,1,Ali,CS101,Intro to Programming,Dr. Smith,A
1,1,Ali,MA201,Linear Algebra,Dr. Jones,B
2,2,Sara,CS101,Intro to Programming,Dr. Smith,A
3,3,Behnam,CS101,Intro to Programming,Dr. Smith,B


### Spotting the Problem: Redundancy

Notice that `"CS101", "Intro to Programming", "Dr. Smith"` is repeated three times. This is **Redundancy** (افزونگی داده) — the same fact (which instructor teaches CS101) is stored in multiple rows.

Why is this bad?

- If Dr. Smith is replaced by a new instructor, you'd have to update **every row** that mentions CS101 — miss one, and now the data **contradicts itself**. This is exactly the Consistency problem Codd's model was built to prevent.
- It also wastes storage repeating the same course name and instructor over and over.

Let's check this with SQL directly:

In [2]:
# Prove the redundancy: how many times is each CourseID's info repeated?
pd.read_sql('''
    SELECT CourseID, CourseName, Instructor, COUNT(*) AS times_repeated
    FROM Enrollments_Bad
    GROUP BY CourseID
''', conn)

,CourseID,CourseName,Instructor,times_repeated
0,CS101,Intro to Programming,Dr. Smith,3
1,MA201,Linear Algebra,Dr. Jones,1


## Step 2: Checking 1NF

Recall the three 1NF rules:

1. Every row must be unique
2. Every value in a column must share the same data type
3. Every field must hold a single, atomic value (no multiple values crammed into one field)

Our `Enrollments_Bad` table actually *does* satisfy 1NF — every row is unique, every column has a consistent type, and no field holds multiple values. **1NF is about row/field shape, not about redundancy across rows** — that's what 2NF exists to catch.

To make the *violation* of 1NF concrete, here's what a rule-3 violation would look like:

In [3]:
# Example of a 1NF VIOLATION: a field holding multiple values at once
bad_1nf_example = pd.DataFrame({
    "StudentID": [1],
    "StudentName": ["Ali"],
    "Courses": ["CS101, MA201, PHY101"],   # <-- not atomic: three values crammed into one field
})
bad_1nf_example

,StudentID,StudentName,Courses
0,1,Ali,"CS101, MA201, PHY101"


This `Courses` field breaks 1NF's third rule directly — you can't easily filter, count, or join on individual courses when they're bundled into a single string. The fix is to give each (student, course) pair its own row, which is exactly what `Enrollments_Bad` already does correctly.

## Step 3: Applying 2NF — Splitting by Functional Dependency

2NF says: **every non-key column must depend on the whole key, not just part of it.**

In `Enrollments_Bad`, the natural key is the pair `(StudentID, CourseID)` — together they identify one enrollment. But look at the functional dependencies:

```text
CourseID  ══════>  CourseName, Instructor      (depends only on CourseID, not on StudentID too)
StudentID ══════>  StudentName                 (depends only on StudentID, not on CourseID too)
```

Both of these are **partial dependencies** — `CourseName` and `Instructor` don't need `StudentID` at all to be determined, and `StudentName` doesn't need `CourseID`. This is precisely what causes the redundancy we found above. The fix: split the table into three, one per functional dependency.

In [4]:
# Table 1: Students (StudentID -> StudentName)
cur.execute('''
CREATE TABLE Students (
    StudentID INTEGER PRIMARY KEY,
    StudentName TEXT NOT NULL
)
''')
cur.executemany("INSERT INTO Students VALUES (?, ?)", [
    (1, "Ali"), (2, "Sara"), (3, "Behnam")
])

# Table 2: Courses (CourseID -> CourseName, Instructor)
cur.execute('''
CREATE TABLE Courses (
    CourseID TEXT PRIMARY KEY,
    CourseName TEXT NOT NULL,
    Instructor TEXT NOT NULL
)
''')
cur.executemany("INSERT INTO Courses VALUES (?, ?, ?)", [
    ("CS101", "Intro to Programming", "Dr. Smith"),
    ("MA201", "Linear Algebra", "Dr. Jones"),
])

# Table 3: Enrollments ((StudentID, CourseID) -> Grade)
cur.execute('''
CREATE TABLE Enrollments (
    StudentID INTEGER,
    CourseID TEXT,
    Grade TEXT,
    PRIMARY KEY (StudentID, CourseID),
    FOREIGN KEY (StudentID) REFERENCES Students(StudentID),
    FOREIGN KEY (CourseID) REFERENCES Courses(CourseID)
)
''')
cur.executemany("INSERT INTO Enrollments VALUES (?, ?, ?)", [
    (1, "CS101", "A"),
    (1, "MA201", "B"),
    (2, "CS101", "A"),
    (3, "CS101", "B"),
])
conn.commit()
print("Three normalized tables created: Students, Courses, Enrollments")

Three normalized tables created: Students, Courses, Enrollments


In [5]:
print("Students:")
display(pd.read_sql("SELECT * FROM Students", conn))

print("\nCourses:")
display(pd.read_sql("SELECT * FROM Courses", conn))

print("\nEnrollments:")
display(pd.read_sql("SELECT * FROM Enrollments", conn))

Students:


,StudentID,StudentName
0,1,Ali
1,2,Sara
2,3,Behnam



Courses:


,CourseID,CourseName,Instructor
0,CS101,Intro to Programming,Dr. Smith
1,MA201,Linear Algebra,Dr. Jones



Enrollments:


,StudentID,CourseID,Grade
0,1,CS101,A
1,1,MA201,B
2,2,CS101,A
3,3,CS101,B


Notice: `"CS101", "Intro to Programming", "Dr. Smith"` now appears **exactly once**, in the `Courses` table — no matter how many students enroll in it. The redundancy is gone. If Dr. Smith is replaced, you update **one row**, in one table, and it's correct everywhere.

## 3NF (Third Normal Form)

Recall the sequence so far:

- **1NF** — fixes the *shape* of a single table (unique rows, consistent types, atomic fields)
- **2NF** — fixes *partial* dependency (a non-key column depending on only part of a composite key)
- **3NF** — fixes **transitive dependency**: a non-key column depending on *another non-key column*, instead of depending directly on the key

**Rule:** A table is in 3NF when it's already in 2NF, and every non-key column depends **only** on the key — not on some other non-key column.


### Example: A Table That Violates 3NF

Consider a student table that also stores department information:

In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.execute('''
CREATE TABLE Students_Bad3NF (
    StudentID INTEGER PRIMARY KEY,
    StudentName TEXT,
    Department TEXT,
    DepartmentBuilding TEXT
)
''')
cur.executemany("INSERT INTO Students_Bad3NF VALUES (?, ?, ?, ?)", [
    (1, "Ali",    "Computer Science", "Tech Building"),
    (2, "Sara",   "Computer Science", "Tech Building"),
    (3, "Behnam", "Mathematics",      "Science Hall"),
])
conn.commit()

pd.read_sql("SELECT * FROM Students_Bad3NF", conn)

,StudentID,StudentName,Department,DepartmentBuilding
0,1,Ali,Computer Science,Tech Building
1,2,Sara,Computer Science,Tech Building
2,3,Behnam,Mathematics,Science Hall


### Why This Violates 3NF

Trace the dependencies:

```text
StudentID  ══════>  Department            (fine — depends on the key)
Department ══════>  DepartmentBuilding    (this is the problem)
```

`DepartmentBuilding` doesn't actually depend on `StudentID` — it depends on `Department`, which is itself just a non-key column. This is a **transitive dependency**: `StudentID → Department → DepartmentBuilding`.

The redundancy this causes is visible directly: `"Computer Science", "Tech Building"` is repeated for every student in that department. If the CS department relocates, you'd need to update every matching row — the same consistency risk 2NF was designed to prevent, just one link further down the chain.

In [9]:
# Fix: split off Department into its own table
cur.execute('''
CREATE TABLE Departments (
    Department TEXT PRIMARY KEY,
    Building TEXT NOT NULL
)
''')
cur.executemany("INSERT INTO Departments VALUES (?, ?)", [
    ("Computer Science", "Tech Building"),
    ("Mathematics",      "Science Hall"),
])

cur.execute('''
CREATE TABLE Students_3NF (
    StudentID INTEGER PRIMARY KEY,
    StudentName TEXT,
    Department TEXT,
    FOREIGN KEY (Department) REFERENCES Departments(Department)
)
''')
cur.executemany("INSERT INTO Students_3NF VALUES (?, ?, ?)", [
    (1, "Ali",    "Computer Science"),
    (2, "Sara",   "Computer Science"),
    (3, "Behnam", "Mathematics"),
])
conn.commit()

print("Students_3NF:")
display(pd.read_sql("SELECT * FROM Students_3NF", conn))
print("\nDepartments:")
display(pd.read_sql("SELECT * FROM Departments", conn))

Students_3NF:


,StudentID,StudentName,Department
0,1,Ali,Computer Science
1,2,Sara,Computer Science
2,3,Behnam,Mathematics



Departments:


,Department,Building
0,Computer Science,Tech Building
1,Mathematics,Science Hall


Now `"Tech Building"` is stored exactly once, in `Departments`. `Students_3NF` only stores which department a student belongs to — the building is looked up via a `JOIN` whenever needed:

In [10]:
pd.read_sql('''
    SELECT s.StudentName, s.Department, d.Building
    FROM Students_3NF s
    JOIN Departments d ON s.Department = d.Department
''', conn)

,StudentName,Department,Building
0,Ali,Computer Science,Tech Building
1,Sara,Computer Science,Tech Building
2,Behnam,Mathematics,Science Hall


### Quick Reference: All Three Normal Forms

| Form | Problem It Fixes | Rule |
|---|---|---|
| **1NF** | Messy table shape | Unique rows, consistent column types, atomic (single-value) fields |
| **2NF** | Partial dependency | Every non-key column depends on the **whole** key, not part of it |
| **3NF** | Transitive dependency | Every non-key column depends **only** on the key, not on another non-key column |

---


## Table Relationships: One-to-Many, Many-to-One, Many-to-Many

Once tables are normalized and split apart, they need to be connected back together — and *how* they connect falls into one of three relationship types. These aren't independent concepts from normalization; they're a direct consequence of it. Splitting `Students_Bad3NF` above already created a relationship between `Students_3NF` and `Departments`.

### 1. One-to-Many (and Many-to-One — Same Relationship, Two Directions)

**One-to-Many** and **Many-to-One** describe the *exact same relationship*, just viewed from opposite sides:

- "**One** Department has **Many** Students" → One-to-Many, viewed from Departments
- "**Many** Students belong to **One** Department" → Many-to-One, viewed from Students

This is precisely the `Departments` ↔ `Students_3NF` relationship above. The rule for implementing it in SQL: **put the foreign key on the "many" side.**

```text
Departments (one)  ◄──────  Students_3NF (many)
   Department (PK)              Department (FK)
```

Let's confirm with a query — one department, many students:

In [11]:
pd.read_sql('''
    SELECT d.Department, d.Building, COUNT(s.StudentID) AS num_students
    FROM Departments d
    LEFT JOIN Students_3NF s ON d.Department = s.Department
    GROUP BY d.Department
''', conn)

,Department,Building,num_students
0,Computer Science,Tech Building,2
1,Mathematics,Science Hall,1


Each department here has more than one student, but each student belongs to exactly one department — that asymmetry is the definition of One-to-Many / Many-to-One.

### More One-to-Many Examples (for pattern recognition)

| "One" side | "Many" side | Real-world relationship |
|---|---|---|
| Customer | Orders | One customer places many orders |
| Author | Books | One author can write many books |
| Manager | Employees | One manager supervises many employees |
| Country | Cities | One country contains many cities |

In every case, the foreign key goes on the "many" table (`Orders.CustomerID`, `Books.AuthorID`, `Employees.ManagerID`, `Cities.CountryID`).

### 2. Many-to-Many

**Many-to-Many** is when records on *both* sides can relate to multiple records on the other side. The clearest example is Students and Courses — this is exactly the `Enrollments` table from the last session:

- One student can take **many** courses
- One course can have **many** students enrolled

Neither side can hold a simple foreign key pointing to the other, because a single column can't reference multiple rows at once. The fix is a **junction table** (also called a bridge or associative table) sitting between the two, holding one row per actual pairing.

In [12]:
cur.execute('''
CREATE TABLE Courses (
    CourseID TEXT PRIMARY KEY,
    CourseName TEXT NOT NULL
)
''')
cur.executemany("INSERT INTO Courses VALUES (?, ?)", [
    ("CS101", "Intro to Programming"),
    ("MA201", "Linear Algebra"),
    ("PHY101", "Physics I"),
])

# The junction table: one row per (student, course) pairing
cur.execute('''
CREATE TABLE Enrollments (
    StudentID INTEGER,
    CourseID TEXT,
    PRIMARY KEY (StudentID, CourseID),
    FOREIGN KEY (StudentID) REFERENCES Students_3NF(StudentID),
    FOREIGN KEY (CourseID) REFERENCES Courses(CourseID)
)
''')
cur.executemany("INSERT INTO Enrollments VALUES (?, ?)", [
    (1, "CS101"), (1, "MA201"), (1, "PHY101"),   # Ali takes 3 courses
    (2, "CS101"),                                  # Sara takes 1 course
    (3, "CS101"), (3, "MA201"),                    # Behnam takes 2 courses
])
conn.commit()

print("Courses:")
display(pd.read_sql("SELECT * FROM Courses", conn))
print("\nEnrollments (the junction table):")
display(pd.read_sql("SELECT * FROM Enrollments", conn))

Courses:


,CourseID,CourseName
0,CS101,Intro to Programming
1,MA201,Linear Algebra
2,PHY101,Physics I



Enrollments (the junction table):


,StudentID,CourseID
0,1,CS101
1,1,MA201
2,1,PHY101
3,2,CS101
4,3,CS101
5,3,MA201


In [13]:
# Proving it's Many-to-Many: each student has multiple courses, AND each course has multiple students
print("Courses per student:")
display(pd.read_sql('''
    SELECT s.StudentName, COUNT(e.CourseID) AS num_courses
    FROM Students_3NF s JOIN Enrollments e ON s.StudentID = e.StudentID
    GROUP BY s.StudentName
''', conn))

print("\nStudents per course:")
display(pd.read_sql('''
    SELECT c.CourseName, COUNT(e.StudentID) AS num_students
    FROM Courses c JOIN Enrollments e ON c.CourseID = e.CourseID
    GROUP BY c.CourseName
''', conn))

Courses per student:


,StudentName,num_courses
0,Ali,3
1,Behnam,2
2,Sara,1



Students per course:


,CourseName,num_students
0,Intro to Programming,3
1,Linear Algebra,2
2,Physics I,1


Both counts exceed 1 in places — confirming the relationship genuinely goes both directions, unlike One-to-Many where only one side could have counts greater than 1.

### More Many-to-Many Examples (for pattern recognition)

| Table A | Table B | Junction Table | Real-world relationship |
|---|---|---|---|
| Students | Courses | Enrollments | A student takes many courses; a course has many students |
| Authors | Books | BookAuthors | A book can have multiple authors; an author writes multiple books |
| Actors | Movies | MovieCast | A movie has many actors; an actor appears in many movies |
| Products | Orders | OrderItems | An order contains many products; a product appears in many orders |

The pattern is always the same: **whenever "many" appears on both sides of a relationship, you need a junction table** — a plain foreign key on either side alone can't represent it.

---

## Summary Diagram: All Three Relationship Types

```text
ONE-TO-MANY / MANY-TO-ONE:
   Departments (1) ─────────< Students (many)
      [foreign key lives on the "many" side]

MANY-TO-MANY:
   Students (many) >──── Enrollments ────< Courses (many)
      [a junction table sits in between, holding foreign keys to both]
```


## Normalization — Step by Step, From Scratch

### Step 1: Start With a Flat, Unnormalized Table

Look at your raw table and ask: does the same information repeat across multiple rows unnecessarily? If yes, it needs normalizing.

**Example — the raw table:**

| StudentID | StudentName | CourseID | CourseName | Instructor | Department | Building |
|---|---|---|---|---|---|---|
| 1 | Ali | CS101 | Intro to Programming | Dr. Smith | CS | Tech Bldg |
| 1 | Ali | MA201 | Linear Algebra | Dr. Jones | CS | Tech Bldg |
| 2 | Sara | CS101 | Intro to Programming | Dr. Smith | CS | Tech Bldg |

### Step 2: Identify the Key Column vs. Non-Key Columns

This is the part you asked about specifically — here's how to actually tell them apart:

**A Key Column is whichever column (or combination of columns) uniquely identifies one row.** Ask: *"If I know only this value, do I know exactly which row I'm talking about?"*

- `StudentID` alone doesn't uniquely identify a row here (Ali has two rows). Neither does `CourseID` alone.
- But the **pair** `(StudentID, CourseID)` together does — no two rows share that same combination. So the key here is the composite key `(StudentID, CourseID)`.

**Everything else is a Non-Key Column** — every column *not* part of that identifying combination: `StudentName`, `CourseName`, `Instructor`, `Department`, `Building`.

**The practical test, column by column:**
1. Can this column repeat for different real-world entities? → it's likely a non-key column.
2. Does this column (alone or combined with others) never repeat, and every other column can be looked up from it? → it's a key column.
3. Write out the dependencies as arrows: `KeyColumn → NonKeyColumn`. If a non-key column's value is fully determined once you know the key, that arrow is valid — this is a **Functional Dependency**.

### Step 3: Apply 1NF

**Rule:** unique rows + consistent data type per column + atomic (single-value) fields.

Check the table above — no field holds multiple values, every row is unique, columns are consistent types. This table already passes 1NF. (A 1NF *violation* would look like a `Courses` field containing `"CS101, MA201"` crammed into one cell — that needs splitting into separate rows first.)

### Step 4: Apply 2NF — Remove Partial Dependency

**Rule:** every non-key column must depend on the **whole** key, not just part of it.

Our key is `(StudentID, CourseID)`. Check each non-key column:

```text
StudentName  ← depends only on StudentID (not CourseID)   → PARTIAL dependency
CourseName   ← depends only on CourseID (not StudentID)   → PARTIAL dependency
Instructor   ← depends only on CourseID (not StudentID)   → PARTIAL dependency
```

Since these depend on only *part* of the composite key, they violate 2NF. **Fix:** split them into separate tables, one per partial dependency:

- `Students(StudentID, StudentName)`
- `Courses(CourseID, CourseName, Instructor, Department, Building)`
- `Enrollments(StudentID, CourseID)` ← what's left, the pure relationship

### Step 5: Apply 3NF — Remove Transitive Dependency

**Rule:** every non-key column must depend **only** on the key — not on another non-key column.

Look inside the new `Courses` table. Its key is `CourseID`. Check the remaining columns:

```text
CourseID → Department            (fine — depends on the key)
Department → Building            (PROBLEM — depends on another non-key column)
```

`Building` doesn't depend on `CourseID` directly — it depends on `Department`, which is itself non-key. This chain, `CourseID → Department → Building`, is a **transitive dependency**. **Fix:** split it out:

- `Courses(CourseID, CourseName, Instructor, Department)`
- `Departments(Department, Building)`

### Final Normalized Schema

```text
Students(StudentID, StudentName)
Departments(Department, Building)
Courses(CourseID, CourseName, Instructor, Department)
Enrollments(StudentID, CourseID)
```

Each fact now lives in exactly one place.

---

## Relationships — How to Identify Which Type Applies

Once tables are split, ask this for **each pair** of tables: *"Can one row on Side A relate to multiple rows on Side B — and vice versa?"*

| Ask this | Answer | Relationship | How to implement |
|---|---|---|---|
| Can one Department have many Students, but each Student belongs to only one Department? | Yes, one-directional "many" | **One-to-Many / Many-to-One** | Put a foreign key on the "many" side (`Students.Department`) |
| Can one Student take many Courses, **and** one Course have many Students? | Yes, "many" on **both** sides | **Many-to-Many** | Create a junction table with foreign keys to both (`Enrollments(StudentID, CourseID)`) |

**Applied to our example:**

- `Departments` ↔ `Courses` → One department has many courses, but each course belongs to one department → **One-to-Many** (foreign key `Courses.Department`)
- `Students` ↔ `Courses` → many students per course AND many courses per student → **Many-to-Many** → needs the `Enrollments` junction table

**The quick rule of thumb:** if a single foreign key column can express the relationship, it's One-to-Many/Many-to-One. If you find yourself needing a foreign key that would have to hold *multiple* values, that's your signal you actually need a junction table — Many-to-Many.